In [1]:
from operator import itemgetter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader 
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_classic.memory import ConversationBufferMemory


llm = ChatOpenAI(model='gpt-4o', temperature=0.1)
memory = ConversationBufferMemory(return_messages=True, memory_key="history")


loader = TextLoader("./files/document.txt")
splitter = CharacterTextSplitter(separator="\n", chunk_size=600, chunk_overlap=100)
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])


chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "history": RunnableLambda(memory.load_memory_variables) | itemgetter("history"),
    }
    | prompt
    | llm
)


def invoke_chain(question):
    print(f"\n--- Question: {question} ---")
    result = chain.invoke({"question": question})
    # Save to memory
    memory.save_context({"input": question}, {"output": result.content})
    print(result.content)


questions = [
    "Is Aaronson guilty?",
    "What message did he write in the table?",
    "Who is Julia?"
]

for q in questions:
    invoke_chain(q)

/var/folders/lc/p9nk00kj5ynb9rq_j5b601zw0000gn/T/ipykernel_31621/4089599783.py:12: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True, memory_key="history")



--- Question: Is Aaronson guilty? ---
According to the context provided, Jones, Aaronson, and Rutherford were charged with crimes and were said to be guilty. However, there was a photograph that disproved their guilt, but it is mentioned that the photograph had never existed and was invented. This suggests that their guilt is questionable, but the narrative claims they were guilty.

--- Question: What message did he write in the table? ---
The context does not provide information about any message being written on a table.

--- Question: Who is Julia? ---
Julia is a person whom the protagonist has a deep emotional connection with. At one point, he cries out her name and expresses love for her. However, in a moment of fear and desperation, he also betrays her by pleading for her to be punished instead of him.
